<a href="https://colab.research.google.com/github/d4nd14z/DataAnalytics/blob/main/PrecioDolar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Análisis de Series de Tiempo**<br/>Variación en los precios del dólar<br/>Daniel Mauricio Díaz Forero<br/>Febrero 2026

### **0. Activación de Google Drive e importación de librerías**

In [71]:
# Activar Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Importar librerías
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### **1. Lectura del Dataset**

In [72]:
ruta_dataset = "/content/drive/MyDrive/Mi Portafolio/precio_dolar/data/Datos históricos USD_COP.csv"
try:
  if not os.path.exists(ruta_dataset):
    raise FileNotFoundError(f"No se encontró el archivo CSV de datos.")
  df = pd.read_csv(ruta_dataset)
except Exception as ex:
  print(f"(ERROR): No se ha podido realizar la lectura de datos.")

### **2. Preparación y Limpieza del Conjunto de Datos**

In [73]:
# Renombrar las columnas del Dataset
df = df.rename(columns={'Fecha': 'fecha', 'Último': 'cierre', 'Apertura':'apertura', 'Máximo':'maximo', 'Mínimo':'minimo', 'Vol.': 'vol', '% var.':'porcentaje_variacion'})

#Convertir a fechas datetime los valores de la columna "fecha"
df["fecha"] = pd.to_datetime(df["fecha"], format='%d.%m.%Y')
df["cierre"] = (df['cierre'].str.replace('.', '', regex=False).str.replace(',', '.', regex=False).astype(float))
df["apertura"] = (df['apertura'].str.replace('.', '', regex=False).str.replace(',', '.', regex=False).astype(float))
df["maximo"] = (df['maximo'].str.replace('.', '', regex=False).str.replace(',', '.', regex=False).astype(float))
df["minimo"] = (df['minimo'].str.replace('.', '', regex=False).str.replace(',', '.', regex=False).astype(float))
df["porcentaje_variacion"] = df['porcentaje_variacion'].str.replace(',', '.').str.rstrip('%').astype(float)

# En un análisis de series de tiempo, la fecha debe ser el índice, no se puede utilizar la fecha como un valor "String"
# Agregamos una nueva columna "id" al dataframe con el contenido de la fecha y convertirmos el campo "id" en el indice del DataFrame.
df.insert(0, 'id', df['fecha'])
df = df.set_index('id').sort_index()

#La columna "vol" no contiene datos relevantes para el análisis.
#Al desconocer su utilidad, no se puede realizar imputación de datos.
#Se toma la decisión de eliminar la columna
df = df.drop(columns=['vol'])

In [74]:
df

,fecha,cierre,apertura,maximo,minimo,porcentaje_variacion
id,,,,,,
2026-01-19,2026-01-19,3659.00,3695.08,3698.39,3649.72,-0.91
2026-01-20,2026-01-20,3679.00,3655.63,3704.40,3655.63,0.55
2026-01-21,2026-01-21,3666.10,3681.23,3683.70,3649.89,-0.35
2026-01-22,2026-01-22,3602.00,3675.41,3676.52,3591.64,-1.75
2026-01-23,2026-01-23,3639.00,3603.50,3667.95,3592.99,1.03
2026-01-26,2026-01-26,3695.00,3644.58,3695.25,3642.81,1.54
2026-01-27,2026-01-27,3622.00,3676.55,3705.54,3622.00,-1.98
2026-01-28,2026-01-28,3672.95,3635.54,3685.42,3616.05,1.41
2026-01-29,2026-01-29,3629.00,3673.62,3680.53,3629.00,-1.20


### **3. Análisis de Datos**<br/>**3.1 Gráfico de Velas (CandleSticks)**

In [75]:
# Crear la figura
fig = go.Figure(data=[go.Candlestick(
    x=df['fecha'],
    open=df['apertura'],
    high=df['maximo'],
    low=df['minimo'],
    close=df['cierre']
)])

# Personalizar el diseño
fig.update_layout(
    title='Análisis de Precios Dólar (Colombia)',
    yaxis_title='Precio ($)',
    xaxis_title='Fecha'
)

fig.show()

### **3.2 Gráfico de Líneas**<br/>**Comportamiento al precio de cierre**

In [78]:
# Comportamiento del precio de cierre
fig = px.line(df, x='fecha', y='cierre', title='Evolución del Precio del Dólar')
fig.show()